In [ ]:
import pandas as pd, numpy as np

%load_ext autoreload
%autoreload 2

!date

# HCES Data Extraction
We have HCES data to inform rice consumption (of PSD-distributed and non-PSD rice) by women and birthing people of 
reproductive age (WBPRA) and U5 children in India. We need to do further investigation as to what percentage of PSD-
distributed rice has been fortified with iron and folate.

HCES data and documentation is saved on Sharepoint here: https://uwnetid.sharepoint.com/:f:/r/sites/ihme_simulation_science_team/Shared%20Documents/Research/LSFF/07_Data/HCES_22_data_files?csf=1&web=1&e=wmxQSD

Item codes for 'Rice' include: 101, 102, and 061. See Section 5.1 of the above documentation for more context.
- Item code 101 signifies rice procured through PDS, using ration card.
- Item code 061 signifies rice procured through PDS, free of charge.
- Item code 102 signifies rice procured/consumed from other sources. (NOTE: Presumably this is unfortified rice.)

We also will need to approximate DHS wealth quintiles in the HCES data, by using similar variables as is used by the DHS to 
calculate wealth quintiles in India (see DHS wealth quintile documentation here: 
https://dhsprogram.com/programming/wealth%20index/India%20DHS%202015-16/India%202015-16%20sps.txt)

In this notebook, we extract the raw data from the HCES website and do some initial processing in order to create files 
that we can further tabulate based on our project needs in a later step. For now, we process these raw data into dataframes 
that have a row for each household/individual with the following columns:
- Whether or not rice was consumed (binary variable - this will be used to calculate coverage percentages later)
- Amount of PDS rice consumed (grams per day - currently in raw HCES, *I think* this is in kg per 2 weeks; this will be used to calculate fortified rice consumption amount) 
- Amount of non-PDS rice consumed (grams per day - currently in raw HCES, *I think* this is in kg per 2 weeks; this will be used to calculate unfortified rice consumption amount) 
- Wealth composite score/quintile 
- Sex/gender (will be used to calculate WBPRA)
- Age (will be used to tabulate age groups)

**2023-24 update:** this notebook now reads the HCES 2023-24 unit-level data. The
acquisition (GHDx NID 583328, help desk ticket DSS-11170) is Nesstar-only; the readable
tab exports live in
`/snfs1/Project/simulation_science/IND_HOUSEHOLD_CONSUMPTION_EXPENDITURE_SURVEY_HCES_2023_2024/`
(see the README.txt there for provenance and verified row counts). Unlike the 2022-23
fixed-width library files, these are tab-delimited with no header row, and the 38-byte
common ID arrives split across the first 16 columns. Field definitions:
`IND_HCES_2023_2024_LAYOUT_Y2025M05D22.XLSX` in the data library's `2023_2024` folder.
Item codes for rice are unchanged (101 PDS ration, 061 PDS free, 102 other) -- but note
PMGKAY made PDS grain free from January 2023, so the consuming-household mix shifts
heavily from 101 to 061 relative to 2022-23.


## Load data

In [ ]:
data_dir = "/snfs1/Project/simulation_science/IND_HOUSEHOLD_CONSUMPTION_EXPENDITURE_SURVEY_HCES_2023_2024"

# The 16 columns the tab exports split the 38-byte fixed-width common ID into, with the
# fixed-width field widths used to reassemble it.
COMMON_ID_COLUMNS = [
    "survey_name",
    "year",
    "fsu_serial_no",
    "sector",
    "state",
    "nss_region",
    "district",
    "stratum",
    "sub_stratum",
    "panel",
    "sub_sample",
    "fod_sub_region",
    "sample_su_no",
    "sample_sub_division_no",
    "second_stage_stratum_no",
    "sample_household_no",
]
COMMON_ID_WIDTHS = [4, 4, 5, 1, 2, 3, 2, 2, 2, 2, 1, 4, 2, 1, 1, 2]

# Row counts from the MoSPI manifest in the package readme. A mismatch means a truncated
# Nesstar export -- they fail silently, and it happened during extraction, so always check.
MANIFEST_ROW_COUNTS = {
    1: 261_953,
    2: 1_107_221,
    3: 261_953,
    5: 12_754_437,
    7: 261_953,
    11: 261_953,
}


def build_common_id(df):
    """Reassemble the 38-character common ID of the 2022-23 fixed-width layout."""
    common_id = ""
    for col, width in zip(COMMON_ID_COLUMNS, COMMON_ID_WIDTHS):
        fill_val = "0"
        side = "left"
        if col in ("sample_su_no", "sample_sub_division_no", "panel"):
            # No clue. (Same convention as the 2022-23 extraction.)
            fill_val = " "
            side = "right"
        values = df[col].fillna(fill_val).str.pad(width, fillchar=fill_val, side=side)
        assert (values.str.len() == width).all()
        common_id = common_id + values
    return common_id


def read_hces_level(level_no, names, keep_id_fields=False, row_filter=None):
    """Read one 2023-24 tab export: no header, common ID split across the first 16 columns."""
    path = f"{data_dir}/IND_HCES_2023_2024_LVL_{level_no:02d}_TAB_Y2026M09D03.TXT"
    kwargs = dict(
        sep="\t",
        header=None,
        names=COMMON_ID_COLUMNS + names,
        # Keep the ID fields as text: they only exist to rebuild the common ID, and
        # numeric parsing would corrupt it (leading zeros, blanks becoming floats).
        dtype={c: str for c in COMMON_ID_COLUMNS},
    )
    if row_filter is None:
        df = pd.read_csv(path, **kwargs)
        n_rows = len(df)
    else:
        # Level 5 is 12.7M rows; filter each chunk before assembling to keep memory sane.
        chunks, n_rows = [], 0
        for chunk in pd.read_csv(path, chunksize=2_000_000, **kwargs):
            n_rows += len(chunk)
            chunks.append(row_filter(chunk))
        df = pd.concat(chunks, ignore_index=True)
    assert n_rows == MANIFEST_ROW_COUNTS[level_no], (level_no, n_rows)
    df.insert(0, "common_id", build_common_id(df))
    if not keep_id_fields:
        df = df.drop(columns=COMMON_ID_COLUMNS)
    return df


In [ ]:
level_1 = read_hces_level(
    1,
    names=[
        "questionnaire_no",
        "level",
        "survey_code",
        "reason_for_substitution_code",
        "multiplier",
    ],
    keep_id_fields=True,
)
level_1


In [ ]:
assert level_1.common_id.is_unique
level_1["common_id"]


In [ ]:
# The 2023-24 HCQ dropped the dwelling building-material, drinking-water and latrine
# questions that 2022-23 asked (they fed the wealth index -- see the note in the wealth
# section), and added ration-card type and PMGKY benefit.
level_3 = read_hces_level(
    3,
    names=[
        "questionnaire_no",
        "level",
        "hh_size",
        "household_member_economic_activity_last_365_days",
        "nco_2015_code",
        "nic_2008_code",
        "max_income_source_last_365_days",
        "major_income_source_self_employment_sector",
        "major_income_regular_wage_sector",
        "major_income_casual_labour_sector",
        "household_type",
        "head_religion",
        "head_social_group",
        "land_ownership",
        "land_type",
        "total_owned_land_area",
        "has_dwelling",
        "dwelling_unit_type",
        "cooking_energy_source",
        "lighting_energy_source",
        "ration_card_type",
        "rural_rent_rate_availability",
        "pmgky_benefit",
        "multiplier",
    ],
)
level_3


In [ ]:
assert level_3.common_id.is_unique

In [ ]:
level_3.cooking_energy_source

In [ ]:
binary_columns = ["land_ownership", "has_dwelling"]

for col in binary_columns:
    level_3[col] = (
        level_3[col]
        .map(
            {
                1: 1.0,
                2: 0.0,
            }
        )
        .astype(float)
    )

In [ ]:
level_3.loc[level_3.land_ownership == 0, "total_owned_land_area"] = 0

In [ ]:
level_3["dwelling_unit_type"] = level_3.dwelling_unit_type.map(
    {
        1: "owned",
        2: "hired",
        3: "others",
    }
)
level_3["house_ownership"] = (level_3.has_dwelling == 1) & (
    level_3.dwelling_unit_type == "owned"
)

In [ ]:
# The 2023-24 HCQ no longer asks about dwelling building materials (2022-23 had
# wall/roof/floor); those wealth-index predictors are unavailable this round.


In [ ]:
level_3["cooking_energy_source"] = level_3.cooking_energy_source.map(
    {
        1: "firewood",
        2: "liquid petroleum gas",
        3: "other natural gas",
        4: "dung",
        5: "kerosene",
        6: "coal",
        7: "gobar",
        8: "other biogas",
        10: "charcoal",
        11: "electricity",
        12: "no cooking",
        9: "other",
    }
)

In [ ]:
level_3["lighting_energy_source"] = level_3.lighting_energy_source.map(
    {
        1: "electricity",
        2: "kerosene",
        3: "other oil",
        4: "gas",
        5: "candle",
        6: "no lighting",
        9: "other",
    }
)

In [ ]:
# The drinking-water-source question was also dropped from the 2023-24 HCQ.


In [ ]:
# The latrine questions (access type and latrine type) were also dropped in 2023-24.


In [ ]:
# The fact that there are a couple very large households (e.g. 20+ members) is a little
# surprising - maybe we can investigate to confirm these are some kind of GQ.
level_3.hh_size.hist()

In [ ]:
level_7 = read_hces_level(
    7,
    names=[
        "questionnaire_no",
        "level",
        "kerosene",
        "lpg_subsidy",
        "num_lpg_subsidy",
        "free_electricity",
        "attend_education",
        "num_attend_govt_ed",
        "num_attend_priv_ed",
        "free_textbooks",
        "num_free_textbooks",
        "free_stationary",
        "num_free_stationary",
        "free_school_bag",
        "num_free_school_bag",
        "other_free_items",
        "num_other_free_items",
        "fee_waiver",
        "num_fee_waiver",
        "pmjay_beneficiary",
        "num_pmjay_beneficiaries",
        "hospitalization_last_365_days",
        "medical_treatment_benefit",
        "num_medical_benefit",
        "medical_benefit_amount",
        "online_purchase_1",
        "online_purchase_2",
        "online_purchase_3",
        "online_purchase_4",
        "online_purchase_5",
        "multiplier",
    ],
)
level_7


In [ ]:
assert level_7.common_id.is_unique

In [ ]:
# 2023-24 dropped the household internet question from the CSQ. Level 2 still asks every
# member about internet use in the last 30 days, so use "any member used the internet"
# as the household indicator. Level 2 is read once here and reused for individual
# consumption below.
household_members_raw = read_hces_level(
    2,
    names=[
        "questionnaire_no",
        "level",
        "person_serial_no",
        "relation_to_head_code",
        "gender",
        "age_years",
        "marital_status_code",
        "highest_education_level_code",
        "total_years_education",
        "internet_use_last_30_days",
        "days_away_from_home_last_30_days",
        "usual_daily_meals",
        "meals_from_school_balwadi_last_30_days",
        "meals_from_employer_last_30_days",
        "meals_others_last_30_days",
        "meals_paid_last_30_days",
        "meals_at_home_last_30_days",
        "member_status_on_revisit",
        "fdq_orig_number",
        "multiplier",
    ],
)

household_internet = (
    (household_members_raw.internet_use_last_30_days == 1)
    .astype(float)
    .groupby(household_members_raw.common_id)
    .max()
    .rename("internet")
)
level_7 = level_7.merge(household_internet, on="common_id", how="left", validate="1:1")


In [ ]:
level_11 = read_hces_level(
    11,
    names=[
        "questionnaire_no",
        "level",
        "online_clothing",
        "online_footwear",
        "online_furniture",
        "online_mobile_handset",
        "online_personal_goods",
        "online_recreation_goods",
        "online_cooking_household",
        "online_crockery",
        "online_sports_goods",
        "online_med_equip",
        "online_bedding",
        "free_laptop",
        "num_free_laptop",
        "free_tablet",
        "num_free_tablet",
        "free_mobile",
        "num_free_mobile",
        "free_bicycle",
        "num_free_bicycle",
        "free_motorcycle",
        "num_free_motorcycle",
        "free_clothing",
        "num_free_clothing",
        "free_footwear",
        "num_free_footwear",
        "other_free_items_dgq",
        "num_other_free_items_dgq",
        "tv",
        "radio",
        "laptop",
        "mobile_handset",
        "bicycle",
        "motorcycle",
        "motor_car",
        "trucks",
        "animal_cart",
        "refrigerator",
        "washing_machine",
        "air_conditioner",
        "multichannel_tv_type",
        "multiplier",
    ],
)
level_11


In [ ]:
assert level_11.common_id.is_unique

In [ ]:
level_11.tv.value_counts(dropna=False)

In [ ]:
possession_columns = [
    "tv",
    "radio",
    "laptop",
    "mobile_handset",
    "bicycle",
    "motorcycle",
    "motor_car",
    "trucks",
    "animal_cart",
    "refrigerator",
    "washing_machine",
    "air_conditioner",
]

In [ ]:
level_11[possession_columns].isnull().mean()

In [ ]:
(level_11[possession_columns] == 0).mean()

In [ ]:
for col in possession_columns:
    level_11[col] = level_11[col].fillna(0.0)

In [ ]:
household_levels = [level_1, level_3, level_7, level_11]

In [ ]:
level_specific_columns = ["level", "questionnaire_no"]


def drop_level_specific(df):
    return df[[c for c in df.columns if c not in level_specific_columns]]


households = drop_level_specific(household_levels[0])

for additional_level in household_levels[1:]:
    common_columns = (set(households.columns) & set(additional_level.columns)) - {
        "common_id"
    }
    households = households.merge(
        drop_level_specific(additional_level),
        on="common_id",
        validate="1:1",
        how="outer",
    )
    for column in common_columns:
        assert (households[f"{column}_x"] == households[f"{column}_y"]).all()
        households[column] = households[f"{column}_x"]
        households = households.drop(columns=[f"{column}_x", f"{column}_y"])

households

In [ ]:
# Bizarrely, I can't find documentation of these codes -- but we know the rent rate is only asked for rural
households.assign(rent_rate=households.rural_rent_rate_availability.notnull()).groupby(
    "sector"
).rent_rate.mean()

In [ ]:
households["sector"] = households.sector.map({"1": "rural", "2": "urban"})
households.sector.value_counts(dropna=False)


In [ ]:
households.isnull().mean().sort_values()

## Calculate wealth quintiles

In order to add wealth stratifications into our HCES data, we will emulate DHS methods for calculating wealth quintiles.
They don't ask something like "how wealthy are you?"
Instead, they ask "do you have a refrigerator?" and then a bunch of other questions like that,
and analyze them into a composite wealth index.
The way they do so is actually pretty weird and interesting, but I won't get into that here.
It is described at https://dhsprogram.com/topics/wealth-index/Wealth-Index-Construction.cfm.

In a first pass, we attempted to use the DHS weights directly.
See [this past version of the notebook](https://github.com/ihmeuw/vivarium_gates_lsff_2026_maternal/blob/12d63b4c5a3b7cb204c4f6eb3de7d6ffe87ce9c3/0100_data_prep/01_extract_hces.ipynb)
for how we started to do this.

However, this was tricky,
because none of the questions asked in HCES are exactly the same, with the same answer options,
as in DHS.
We could sort of use proxies and fudge things, but it was pretty rough.
Here, I have instead used the same *methods* used in the DHS to make an entirely new
model of the latent wealth variable.
I followed the steps described here: https://dhsprogram.com/programming/wealth%20index/Steps_to_constructing_the_new_DHS_Wealth_Index.pdf

However, even this is pretty underwhelming. Probably a better strategy to try in the future would be to make a more complex and
precise crosswalk between DHS and HCES questions/responses, then train a model in DHS to predict the DHS wealth quintile based
only on information available in HCES, and finally predict HCES.

DHS wealth index variables include: 
- Source of drinking water
- Type of toilet facility
- Electricity
- Mattress
- Pressure cooker
- Chair
- Cot or bed
- Table
- Electric fan
- Radio or transistor
- Black and white television
- Colour television
- Sewing machine
- Mobile telephone
- Telephone (non-mobile)
- Internet
- Computer
- Refrigerator
- Air conditioner/cooler
- Washing machine
- Watch or clock
- Bicycle
- Motorcycle or Scooter
- Animal-drawn cart
- Car
- Water pump
- Thresher
- Tractor
- Type of cooking fuel
- Main material of floor
- Main roof material
- Main wall material
- Hectares for agricultural land
- Out of this land, how much is irrigated?
- Cows / bulls / buffaloes
- Camels
- Horses / donkeys / mules
- Goats
- Sheep
- Chickens / ducks
- Bank account
- Members per sleeping room
- Compute urban and rural variables coded (1/0) for filters later
- Toilet facility by shared/not shared
- Land area by units - if there are separate units - need to convert them to one unit

HCES variables:
- Type of land owned
- What is the total area of all owned (owned and possessed or leased out) land (within the country) by the household as on the date of survey (area in acre)?(upto two places of decimal) 
- Basic building Material used for major portion of the wall of the dwelling Unit
- Basic building Material used for construction of the major portion of the outer exposed part of the roof of the dwelling unit
- Basic Building Material used for construction of the major portion of the floor of the dwelling Unit
- Source of Drinking Water (Last 365 days)
- Type of latrine in which the household has access
- Primary source of energy of the household for cooking
- Household has internet facility as on the date of the survey
- Whether household possessed one or more item as on the date of the survey- Television 
- Whether household possessed one or more item as on the date of the survey- Radio
- Whether household possessed one or more item as on the date of the survey - Laptop/PC
- Whether household possessed one or more item as on the date of the survey- Mobile handset
- Whether household possessed one or more item as on the date of the survey- Bicycle
- Whether household possessed one or more item as on the date of the survey- Motorcycle, scooter 
- Whether household possessed one or more item as on the date of the survey- Motor car/jeep/van
- Whether household possessed one or more item as on the date of the survey- Trucks
- Whether household possessed one or more item as on the date of the survey - Animal cart
- Whether household possessed one or more item as on the date of the survey- Refrigerator
- Whether household possessed one or more item as on the date of the survey- Washing machine
- Whether household possessed one or more item as on the date of the survey- Air conditioner/air cooler 
- Type of multichannel television facility is used by the household as on the date of the survey

Crosswalk spreadsheet saved here: https://uwnetid.sharepoint.com/sites/ihme_simulation_science_team/_layouts/15/doc.aspx?sourcedoc={eddcbf51-3462-47d7-b847-79edea5046c6}&action=edit

**2023-24 note:** the HCQ dropped the dwelling building-material, drinking-water and
latrine questions, so those predictors are no longer available; the index is built from
the surviving predictors (land, dwelling ownership, cooking and lighting energy,
internet, and the twelve possessions). Internet now comes from level 2 (any member used
the internet in the last 30 days) instead of the dropped household-level CSQ question.
Quintile assignments therefore shift slightly beyond pure survey change; the ration-card
and PMGKY questions added in 2023-24 were deliberately left out of the index because
they are means-tested program outcomes (and correlate with our PDS exposure).


In [ ]:
wealth_relevant_info = households[
    [
        "sector",
        "hh_size",
        "multiplier",
        # Section/level 4
        "land_ownership",
        # 'total_owned_land_area',
        "has_dwelling",
        "house_ownership",
        "cooking_energy_source",
        "lighting_energy_source",
        # Section 4.2 (from level 2 in 2023-24; see above)
        "internet",
        # Section 4.3
        *possession_columns,
    ]
].copy()
wealth_relevant_info


In [ ]:
wealth_relevant_info.isnull().mean()

In [ ]:
wealth_relevant_info.nunique()

In [ ]:
columns_to_convert = [
    "cooking_energy_source",
    "lighting_energy_source",
]

for column in columns_to_convert:
    # Get dummy variables for the current column
    dummies = pd.get_dummies(wealth_relevant_info[column], prefix=column)

    # Drop the original column from hces_df
    wealth_relevant_info = wealth_relevant_info.drop(column, axis=1)

    # Join the dummy variables to the original DataFrame
    wealth_relevant_info = pd.concat([wealth_relevant_info, dummies], axis=1)


In [ ]:
wealth_predictors = [
    c
    for c in wealth_relevant_info.columns
    if c not in ("sector", "hh_size", "multiplier")
]

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=1)
pca.fit(
    wealth_relevant_info.drop(columns=["sector", "multiplier", "hh_size"]).dropna(
        how="any"
    )
)

In [ ]:
pd.DataFrame(
    {
        "column": wealth_relevant_info.drop(
            columns=["sector", "multiplier", "hh_size"]
        ).columns,
        "scores": pd.Series(pca.components_[0]).astype(float),
    }
).sort_values("scores")

In [ ]:
wealth_relevant_info["common_score"] = pca.transform(
    wealth_relevant_info.drop(columns=["sector", "multiplier", "hh_size"])
)

In [ ]:
wealth_relevant_info.sector.value_counts()

In [ ]:
scores = {}

for sector in ["urban", "rural"]:
    print(sector)
    sector_rows = wealth_relevant_info.sector == sector

    pca = PCA(n_components=1)
    pca.fit(wealth_relevant_info[sector_rows][wealth_predictors].dropna(how="any"))

    display(
        pd.DataFrame(
            {
                "column": wealth_predictors,
                "scores": pd.Series(pca.components_[0]).astype(float),
            }
        ).sort_values("scores")
    )

    wealth_relevant_info.loc[sector_rows, f"{sector}_score"] = pca.transform(
        wealth_relevant_info[sector_rows][wealth_predictors]
    )

In [ ]:
import statsmodels.api as sm

for sector in ["urban", "rural"]:
    print(sector)
    sector_rows = wealth_relevant_info.sector == sector

    mod = sm.OLS(
        wealth_relevant_info[sector_rows][["common_score"]],
        sm.add_constant(wealth_relevant_info[sector_rows][[f"{sector}_score"]]),
    )
    fii = mod.fit()
    print(fii.summary2())

    wealth_relevant_info.loc[sector_rows, "combined_score"] = fii.predict(
        sm.add_constant(wealth_relevant_info[sector_rows][[f"{sector}_score"]])
    )

In [ ]:
wealth_relevant_info.combined_score.hist(bins=500)

In [ ]:
wealth_relevant_info["weighted_household_size"] = (
    wealth_relevant_info.hh_size * wealth_relevant_info.multiplier
)

from statsmodels.stats.weightstats import DescrStatsW

wq = DescrStatsW(
    data=wealth_relevant_info.combined_score,
    weights=wealth_relevant_info.weighted_household_size,
)
quintile_cutoffs = wq.quantile(probs=np.linspace(0, 1, 6))
quintile_cutoffs

In [ ]:
wealth_relevant_info["wealth_quintile"] = pd.cut(
    wealth_relevant_info.combined_score,
    quintile_cutoffs,
    labels=[1, 2, 3, 4, 5],
    include_lowest=True,
)
wealth_relevant_info["wealth_quintile"]

In [ ]:
assert wealth_relevant_info.wealth_quintile.notnull().all()

In [ ]:
wealth_relevant_info.groupby(
    "wealth_quintile", observed=True
).multiplier.sum() / wealth_relevant_info.multiplier.sum()

In [ ]:
for column in wealth_predictors:
    print(column)
    print(
        wealth_relevant_info.groupby(
            ["wealth_quintile", column], observed=True
        ).multiplier.sum()
        / wealth_relevant_info.groupby("wealth_quintile").multiplier.sum()
    )

In [ ]:
# Whoa, owning land or a house is uncorrelated?!
india_dhs = pd.read_stata(
    "/snfs1/DATA/DHS_PROG_DHS/IND/2015_2016/IND_DHS7_2015_2016_HH_IAHR74FL_Y2018M12D06.DTA",
    columns=["hv005", "hv270", "sh46"],
).rename(columns={"hv005": "weight", "hv270": "wealth_quintile", "sh46": "owns_house"})

In [ ]:
india_dhs.groupby(
    ["wealth_quintile", "owns_house"], observed=True
).weight.sum() / india_dhs.groupby("wealth_quintile").weight.sum()

In [ ]:
# Bizarre. Even in DHS it's basically uncorrelated, although in HCES it seems to be reversed.

In [ ]:
households["wealth_quintile"] = wealth_relevant_info["wealth_quintile"]

In [ ]:
assert households.wealth_quintile.notnull().all()

## Household rice consumption

Item codes for 'Rice' include: 101, 102, and 061. See Section 5.1 of the above documentation for more context.

- Item code 101 signifies rice procured through PDS, using ration card.
- Item code 061 signifies rice procured through PDS, free of charge.
- Item code 102 signifies rice procured/consumed from other sources. (NOTE: Presumably this is unfortified rice.)

Units: kg/30 days.

In [ ]:
consumption = read_hces_level(
    5,
    names=[
        "q_num",
        "level",
        "item_code",
        "consumption_amt_home_produce",
        "consumption_value_home_produce",
        "total_consumption_amt",
        "total_consumption_value",
        "source",
        "multiplier",
    ],
    row_filter=lambda df: df[df.item_code.isin([61, 101, 102])],
)
consumption


In [ ]:
consumption["item_code"] = consumption.item_code.map(
    {
        61: "pds free rice",
        101: "pds paid rice",
        102: "other rice",
    }
)
consumption["is_pds"] = consumption.item_code.isin(["pds free rice", "pds paid rice"])

In [ ]:
consumption["source"] = consumption.source.map(
    {
        1: "only purchase",
        2: "only home-grown",
        3: "both purchase and home-grown",
        4: "only free collection",
        5: "only exchange",
        6: "only gifts",
        9: "other",
    }
)

In [ ]:
consumption["total_consumption_amt"] = consumption.total_consumption_amt.astype(float)
consumption["consumption_amt_home_produce"] = (
    consumption.consumption_amt_home_produce.astype(float)
)

In [ ]:
# I assume for these high-missingness columns, null means 0
consumption.isnull().mean()

In [ ]:
(consumption == 0).mean()

In [ ]:
consumption = consumption.fillna(0)

In [ ]:
consumption["pds_rice"] = np.where(
    consumption.is_pds,
    consumption.total_consumption_amt,
    0.0,
)

In [ ]:
consumption["non_pds_rice_purchased"] = np.where(
    consumption.is_pds,
    0.0,
    np.where(
        consumption.source.isin(["only purchase", "both purchase and home-grown"]),
        consumption.total_consumption_amt - consumption.consumption_amt_home_produce,
        0.0,
    ),
)

In [ ]:
consumption["non_pds_rice_non_purchased"] = (
    consumption.total_consumption_amt
    - consumption.pds_rice
    - consumption.non_pds_rice_purchased
)

In [ ]:
consumption = consumption.groupby("common_id")[
    ["pds_rice", "non_pds_rice_purchased", "non_pds_rice_non_purchased"]
].sum()
consumption

In [ ]:
consumption["rice"] = consumption.sum(axis=1)
consumption["proportion_non_pds_purchased"] = (
    consumption.non_pds_rice_purchased / (consumption.rice - consumption.pds_rice)
).clip(0, 1)
consumption

In [ ]:
len(consumption) / len(households)

In [ ]:
(households.merge(consumption, on="common_id", how="left").rice > 0).mean()

In [ ]:
consumption = consumption.reindex(households.common_id).fillna(0).reset_index()
consumption

In [ ]:
(households.merge(consumption, on="common_id", how="left").rice > 0).mean()

## Individual rice consumption

We distribute the rice to the individuals according to the number of meals they have eaten at home.
We assume that children eating school lunches are getting fortified rice, as the scale-up has also
happened through that program.
Otherwise, we assume that meals eaten away from the home use rice and fortified rice at the same rate
as meals at home.

In [ ]:
# Level 2 was read once in the wealth section (for the internet indicator); reuse it.
household_members = household_members_raw[
    ["common_id", "gender", "age_years", "multiplier"]
    + [c for c in household_members_raw.columns if "meals" in c]
]
household_members


In [ ]:
# My interpretation: null basically is equivalent to 0 for these high-missingness columns...
household_members.filter(like="meals").isnull().mean()

In [ ]:
# ... even though there are actual zeros!
(household_members.filter(like="meals") == 0).mean()

In [ ]:
for col in [
    "meals_from_school_balwadi_last_30_days",
    "meals_from_employer_last_30_days",
    "meals_others_last_30_days",
    "meals_paid_last_30_days",
]:
    household_members[col] = household_members[col].fillna(0)

In [ ]:
household_members.isnull().mean()

In [ ]:
# Complete-case analysis
household_members = household_members.dropna(how="any").copy()
household_members

In [ ]:
household_members["gender"] = household_members.gender.map(
    {
        1: "Male",
        2: "Female",
        3: "Transgender",
    }
)

In [ ]:
household_members = household_members.merge(
    households[["common_id", "wealth_quintile"]],
    on="common_id",
    validate="m:1",
    how="left",
)

In [ ]:
assert household_members.wealth_quintile.notnull().all()

In [ ]:
household_members["meals"] = (
    household_members.meals_at_home_last_30_days.fillna(0)
    + household_members.meals_from_employer_last_30_days.fillna(0)
    + household_members.meals_from_school_balwadi_last_30_days.fillna(0)
    + household_members.meals_from_employer_last_30_days.fillna(0)
    + household_members.meals_paid_last_30_days.fillna(0)
    + household_members.meals_others_last_30_days.fillna(0)
)

In [ ]:
household_members.plot(kind="scatter", x="usual_daily_meals", y="meals", alpha=0.01)

In [ ]:
household_meals = household_members.groupby(
    "common_id"
).meals_at_home_last_30_days.sum()
household_meals

In [ ]:
consumption = consumption.merge(
    household_meals, on="common_id", validate="1:1", how="left"
)
consumption

In [ ]:
consumption.isnull().mean()

In [ ]:
consumption = consumption.dropna(how="any").copy()
consumption

In [ ]:
consumption["rice_per_meal_kg"] = np.where(
    consumption.rice == 0,
    0.0,
    consumption.rice / consumption.meals_at_home_last_30_days,
)

In [ ]:
# In a small number of cases, rice consumption is reported despite very few or no meals at home
impossible_rice_consumption = (consumption.rice > 0) & (
    consumption.meals_at_home_last_30_days == 0
)
consumption[impossible_rice_consumption]

In [ ]:
# This must be wrong somehow, so we clip it
consumption["rice_per_meal_kg"] = consumption["rice_per_meal_kg"].clip(
    upper=np.percentile(consumption.rice_per_meal_kg, 95)
)

In [ ]:
consumption[consumption.rice_per_meal_kg.notnull()].sort_values("rice_per_meal_kg")
consumption.rice_per_meal_kg.hist()

In [ ]:
consumption["pds_rice_per_meal_kg"] = consumption.rice_per_meal_kg * (
    consumption.pds_rice / consumption.rice
)
consumption.pds_rice_per_meal_kg.hist()

In [ ]:
household_members = household_members.merge(
    consumption.reset_index()[
        [
            "common_id",
            "rice_per_meal_kg",
            "pds_rice_per_meal_kg",
            "proportion_non_pds_purchased",
        ]
    ],
    on="common_id",
    how="left",
    validate="m:1",
)
household_members

In [ ]:
household_members.isnull().mean()

In [ ]:
household_members["government_meals_away_from_home"] = (
    household_members.meals_from_school_balwadi_last_30_days
)
household_members["non_government_meals_away_from_home"] = (
    household_members.meals
    - household_members.meals_at_home_last_30_days
    - household_members.government_meals_away_from_home
)
household_members["rice"] = household_members.meals * household_members.rice_per_meal_kg
household_members["government_rice"] = (
    household_members.meals_at_home_last_30_days
    * household_members.pds_rice_per_meal_kg
    + (
        household_members.government_meals_away_from_home
        * household_members.rice_per_meal_kg
    )
    + (
        household_members.non_government_meals_away_from_home
        * household_members.pds_rice_per_meal_kg
    )  # Assume same as at-home meals
)
household_members["proportion_government"] = (
    household_members.government_rice / household_members.rice
).clip(0, 1)
# NOTE: For now, we say only government rice is fortifiable!
household_members["proportion_fortifiable"] = household_members["proportion_government"]
# (
#     # It's assumed that all non-government rice you consume is fortifiable if it was purchased; we use your household's rate of
#     # purchasing vs growing for non-PDS rice for this.
#     # All government rice is fortifiable.
#     (
#         household_members.government_rice
#         + (
#             household_members.proportion_non_pds_purchased
#             * (household_members.rice - household_members.government_rice)
#         )
#     )
#     / household_members.rice
# )

In [ ]:
household_members.isnull().mean()

In [ ]:
# Convert kg/30 days to g/day
household_members["rice_g_per_day"] = household_members.rice / 30 * 1_000

In [ ]:
household_members["sex"] = household_members.gender.replace(
    {"Transgender": np.nan}
)  # We don't know the sex of transgender people in the survey :(
age_bin_edges = [0, 5, 15, 30, 50, 125]
age_group = pd.IntervalIndex(
    pd.cut(household_members.age_years, age_bin_edges, right=False, include_lowest=True)
)
household_members["age_start"] = age_group.left
household_members["age_end"] = age_group.right

In [ ]:
# https://stackoverflow.com/a/2415343/ with some tweaks
def weighted_avg_and_std(values, weights):
    """
    Return the weighted average and standard deviation.

    They weights are in effect first normalized so that they
    sum to 1 (and so they must not all be 0).

    values, weights -- NumPy ndarrays with the same shape.
    """
    is_nan = np.isnan(values)
    values = values[~is_nan]
    # The tab exports parse the multiplier as int64, and weights.sum()**2 overflows
    # int64 in the effective-sample-size line; promote to float.
    weights = weights[~is_nan].astype(float)
    average = np.average(values, weights=weights)
    # Fast and numerically precise:
    variance = np.average((values - average) ** 2, weights=weights)
    return pd.Series(
        {
            "mean": average,
            "sd": np.sqrt(variance),
            # https://ngreifer.github.io/WeightIt/reference/ESS.html
            "effective_sample_size": (weights.sum() ** 2) / (weights**2).sum(),
        }
    )

In [ ]:
results_dir = "../results/"

In [ ]:
import pathlib


def save_data(series, name):
    path = f"{results_dir}/{name}/india.csv"
    pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
    (
        series.rename("value")
        .reset_index()
        .assign(vehicle_name="rice")
        .to_csv(path, index=False)
    )


def save_for_each_fortificant(series, name):
    save_data(series, f"iron/{name}")
    save_data(series, f"folate/{name}")

In [ ]:
weighted_avg_and_std(
    household_members.rice_g_per_day > 0, weights=household_members.multiplier
)

In [ ]:
(
    household_members.groupby("wealth_quintile")
    .apply(
        lambda df: weighted_avg_and_std(df.rice_g_per_day > 0, weights=df.multiplier)
    )
    .sort_index()
)

In [ ]:
# NOTE: We do not have pregnancy status in this survey, so we assume it has no effect on consumption!
any_consumption = (
    household_members.groupby(
        ["sex", "age_start", "age_end", "wealth_quintile"], observed=True
    )
    .apply(
        lambda df: weighted_avg_and_std(
            np.where(df.rice_g_per_day.notnull(), df.rice_g_per_day > 0, np.nan),
            weights=df.multiplier,
        )
    )
    .sort_index()
)
any_consumption

In [ ]:
save_data(any_consumption["mean"], "rice/vehicle_consumption/any")

In [ ]:
weighted_avg_and_std(
    household_members.rice_g_per_day, weights=household_members.multiplier
)

In [ ]:
(
    household_members.groupby("wealth_quintile")
    .apply(lambda df: weighted_avg_and_std(df.rice_g_per_day, weights=df.multiplier))
    .sort_index()
)

In [ ]:
consumption_amount = (
    household_members.groupby(
        ["sex", "age_start", "age_end", "wealth_quintile"], observed=True
    )
    .apply(lambda df: weighted_avg_and_std(df.rice_g_per_day, weights=df.multiplier))
    .sort_index()
)
consumption_amount

In [ ]:
save_data(consumption_amount["mean"], "rice/vehicle_consumption/amount/mean")
save_data(consumption_amount["sd"], "rice/vehicle_consumption/amount/sd")

In [ ]:
# NOTE: Government rice fortification scale-up is still in progress.
# According to this dashboard (https://impds.nic.in/sale/stateUnautmated?month=1&year=2024) it is
# around 70% in 2024. We say that it can't get above 80% due to compliance issues.
# We apply maximum individual heterogeneity, meaning that the people who are missed
# due to compliance are *completely* missed.
GOVERNMENT_BASELINE_COVERAGE = 0.8

In [ ]:
weighted_avg_and_std(
    GOVERNMENT_BASELINE_COVERAGE * household_members.proportion_government,
    weights=household_members.multiplier,
)

In [ ]:
(
    household_members.groupby("wealth_quintile")
    .apply(
        lambda df: weighted_avg_and_std(
            GOVERNMENT_BASELINE_COVERAGE * df.proportion_government,
            weights=df.multiplier,
        )
    )
    .sort_index()
)

In [ ]:
any_fortification = (
    household_members.groupby(
        ["sex", "age_start", "age_end", "wealth_quintile"], observed=True
    )
    .apply(
        lambda df: weighted_avg_and_std(
            np.where(
                df.proportion_government.notnull(),
                GOVERNMENT_BASELINE_COVERAGE * (df.proportion_government > 0),
                np.nan,
            ),
            weights=df.multiplier,
        )
    )
    .sort_index()
)
any_fortification

In [ ]:
save_for_each_fortificant(
    any_fortification["mean"], "rice/baseline_fortification/any_coverage"
)

In [ ]:
full_fortification = (
    household_members.groupby(
        ["sex", "age_start", "age_end", "wealth_quintile"], observed=True
    )
    .apply(
        lambda df: weighted_avg_and_std(
            np.where(
                df.proportion_government.notnull(),
                GOVERNMENT_BASELINE_COVERAGE * (df.proportion_government == 1),
                np.nan,
            ),
            weights=df.multiplier,
        )
    )
    .sort_index()
)
full_fortification

In [ ]:
save_for_each_fortificant(
    full_fortification["mean"], "rice/baseline_fortification/full_coverage"
)

In [ ]:
# Remember, we've assumed maximum heterogeneity here, so the people who are partially covered
# have *all* of their government rice fortified.
partial_fortification_amount = (
    household_members[
        (household_members.proportion_government > 0)
        & (household_members.proportion_government < 1)
    ]
    .groupby(["sex", "age_start", "age_end", "wealth_quintile"], observed=True)
    .apply(
        lambda df: weighted_avg_and_std(df.proportion_government, weights=df.multiplier)
    )
    .sort_index()
)
partial_fortification_amount

In [ ]:
save_for_each_fortificant(
    partial_fortification_amount["mean"],
    "rice/baseline_fortification/partial_coverage_amount/mean",
)
save_for_each_fortificant(
    partial_fortification_amount["sd"],
    "rice/baseline_fortification/partial_coverage_amount/sd",
)

In [ ]:
weighted_avg_and_std(
    household_members.proportion_government, weights=household_members.multiplier
)

In [ ]:
proportion_government = (
    household_members.groupby(
        ["sex", "age_start", "age_end", "wealth_quintile"], observed=True
    )
    .apply(
        lambda df: weighted_avg_and_std(df.proportion_government, weights=df.multiplier)
    )
    .sort_index()
)
proportion_government

In [ ]:
# Save locally for further scaling in the extraction notebook
proportion_government["mean"].rename("value").to_csv(
    "india_proportion_government_rice.csv"
)

In [ ]:
(
    household_members.groupby("wealth_quintile")
    .apply(
        lambda df: weighted_avg_and_std(
            df.proportion_fortifiable, weights=df.multiplier
        )
    )
    .sort_index()
)

In [ ]:
fortifiability_disparities = (
    household_members.groupby(
        ["sex", "age_start", "age_end", "wealth_quintile"], observed=True
    )
    .apply(
        lambda df: weighted_avg_and_std(
            df.proportion_fortifiable, weights=df.multiplier
        )
    )
    .sort_index()
)
fortifiability_disparities

In [ ]:
# Save locally for further scaling in the extraction notebook
fortifiability_disparities["mean"].rename("value").to_csv(
    "india_rice_fortifiability_disparities.csv"
)